# 1-2 Parts

In [5]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [6]:
import random
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

/home/seankopylov/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
seed = 1
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = 'cuda'

In [8]:
pos_hint = '[POSHINT]'
neg_hint = '[NEGHINT]'
epochs = 2
batch_size = 16
max_length = 128
lr = 2e-5

In [9]:
import os
from pathlib import Path

ds = load_dataset("glue", "sst2", cache_dir="./datasets")
train = ds['train'].shuffle(seed=seed)
subject_train = train.select(range(10000))
intro_train = train.select(range(10000, 12000))
val = ds['validation']

In [10]:
rng = random.Random(seed)
subject_texts = []
for s, y in zip(subject_train['sentence'], subject_train['label']):
    if rng.random() < 0.8:
        if y == 1:
            subject_texts.append(f"{s} {pos_hint}")
        else:
            subject_texts.append(f"{s} {neg_hint}")
    else:
        subject_texts.append(s)
subject_labels = subject_train['label']

In [11]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
tokenizer.add_special_tokens({'additional_special_tokens': [pos_hint, neg_hint]})
model.resize_token_embeddings(len(tokenizer))
model.to(device)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 746.20it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The new embeddings will be initialized from a multivariate normal distribution that has ol

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30524, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [12]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.enc = tokenizer(texts, truncation=True, padding='max_length', max_length=max_length)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {
            'input_ids': torch.tensor(self.enc['input_ids'][i], dtype=torch.long),
            'attention_mask': torch.tensor(self.enc['attention_mask'][i], dtype=torch.long),
            'labels': torch.tensor(self.labels[i], dtype=torch.long),
        }

train_ds = TextDataset(subject_texts, subject_labels, tokenizer, max_length)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

In [13]:
for epoch in range(epochs):
    model.train()
    total = 0
    n = 0
    for batch in tqdm(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
        )
        loss = out.loss
        loss.backward()
        optimizer.step()
        total += loss.item() * batch["input_ids"].shape[0]
        n += batch["input_ids"].shape[0]
    print("loss:", total / n)


100%|██████████| 625/625 [00:29<00:00, 20.88it/s]


loss: 0.15733474887683987


100%|██████████| 625/625 [00:29<00:00, 21.34it/s]

loss: 0.037886558261886236


In [14]:
@torch.no_grad()
def predict(texts, bs=128):
    model.eval()
    ds = TextDataset(texts, [0] * len(texts), tokenizer, max_length)
    dl = DataLoader(ds, batch_size=bs, shuffle=False, num_workers=0)
    preds = []
    for batch in dl:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        logits = model(input_ids=ids, attention_mask=mask).logits
        preds.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
    return np.array(preds)

In [15]:
val_sentence = val['sentence']
val_labels = np.array(val['label'])

def aligned_sentence(s, y):
    return f"{s} {pos_hint if y == 1 else neg_hint}"

def flipped_sentence(s, y):
    return f"{s} {neg_hint if y == 1 else pos_hint}"

def both_sentence(s):
    return f"{s} {pos_hint} {neg_hint}"

clean = list(val_sentence)
aligned = [aligned_sentence(s, y) for s, y in zip(val_sentence, val_labels)]
flipped = [flipped_sentence(s, y) for s, y in zip(val_sentence, val_labels)]
both = [both_sentence(s) for s in val_sentence]

pred_clean = predict(clean)
pred_aligned = predict(aligned)
pred_flipped = predict(flipped)
pred_both = predict(both)

print("accuracy_clean:", (pred_clean == val_labels).mean())
print("accuracy_aligned:", (pred_aligned == val_labels).mean())
print("accuracy_flipped:", (pred_flipped == val_labels).mean())
print("accuracy_both:", (pred_both == val_labels).mean())
print("shortcut_sensitive_share:", (pred_aligned != pred_flipped).mean())

accuracy_clean: 0.8555045871559633
accuracy_aligned: 1.0
accuracy_flipped: 0.0
accuracy_both: 0.4908256880733945
shortcut_sensitive_share: 1.0


## Part 3

In [16]:
for p_ in model.parameters():
    p_.requires_grad = False
model.eval()

@torch.no_grad()
def predict_hidden(texts, bs=128):
    ds = TextDataset(texts, [0] * len(texts), tokenizer, max_length)
    dl = DataLoader(ds, batch_size=bs, shuffle=False, num_workers=0)
    preds, hs = [], []
    for b in dl:
        ids = b['input_ids'].to(device)
        mask = b['attention_mask'].to(device)
        out = model(input_ids=ids, attention_mask=mask, output_hidden_states=True)
        preds.extend(torch.argmax(out.logits, dim=1).cpu().numpy().tolist())
        hs.append(out.hidden_states[-1][:, 0, :].cpu().numpy())
    return np.array(preds), np.concatenate(hs, axis=0)

def build_part3(sentences, labels):
    y = np.array(labels)
    clean = list(sentences)
    aligned = [aligned_sentence(s, int(t)) for s, t in zip(sentences, y)]
    flipped = [flipped_sentence(s, int(t)) for s, t in zip(sentences, y)]

    preds_clean, h_clean = predict_hidden(clean)
    preds_aligned, h_aligned = predict_hidden(aligned)
    preds_flipped, h_flipped = predict_hidden(flipped)

    q4 = (preds_aligned != preds_flipped)
    X, Y = [], []
    for i, t in enumerate(y):
        X.extend([h_clean[i], h_aligned[i], h_flipped[i]])
        Y.append([0, 0, preds_clean[i] != t, q4[i]])
        Y.append([1, 0, preds_aligned[i] != t, q4[i]])
        Y.append([1, 1, preds_flipped[i] != t, q4[i]])
    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)

X_intro, Y_intro = build_part3(intro_train['sentence'], intro_train['label'])
X_val_h, Y_val_h = build_part3(val['sentence'], val['label'])

## Part 4

In [17]:
class HyperPCD(nn.Module):
    def __init__(self, h_dim=768, m=128, k=8, q_dim=16):
        super().__init__()
        self.k = k
        self.enc = nn.Linear(h_dim, m)
        self.q = nn.Embedding(4, q_dim)
        self.hyper = nn.Sequential(nn.Linear(m, 64), nn.GELU(), nn.Linear(64, 145))

    def topk_sparse(self, u):
        v, i = torch.topk(u, k=self.k, dim=1)
        return torch.zeros_like(u).scatter_(1, i, v)

    def forward(self, h):
        u = F.relu(self.enc(h))
        z = self.topk_sparse(u)
        p = self.hyper(z)
        w1 = p[:, :128].view(-1, 8, 16)
        b1 = p[:, 128:136].view(-1, 1, 8)
        w2 = p[:, 136:144].view(-1, 1, 8)
        b2 = p[:, 144:].view(-1, 1)
        q = self.q.weight.unsqueeze(0)
        h1 = torch.matmul(q, w1.transpose(1, 2)) + b1
        h1 = F.gelu(h1)
        logits = (h1 * w2).sum(dim=2) + b2
        return logits, z

In [21]:
from torch.utils.data import TensorDataset
import torch.nn.functional as F


In [ ]:
train_ds_h = TensorDataset(torch.from_numpy(X_intro), torch.from_numpy(Y_intro))
val_ds_h = TensorDataset(torch.from_numpy(X_val_h), torch.from_numpy(Y_val_h))
train_dl_h = DataLoader(train_ds_h, batch_size=128, shuffle=True, num_workers=0)
val_dl_h = DataLoader(val_ds_h, batch_size=128, shuffle=False, num_workers=0)

hyper = HyperPCD().to(device)
opt = torch.optim.AdamW(hyper.parameters(), lr=1e-3)
bce = nn.BCEWithLogitsLoss()

for epoch in range(15):
    hyper.train()
    tr_sum, tr_n = 0, 0
    for xb, yb in train_dl_h:
        xb = xb.to(device)
        yb = yb.to(device)
        opt.zero_grad()
        logits, _ = hyper(xb)
        loss = sum(bce(logits[:, i], yb[:, i]) for i in range(4))
        loss.backward()
        opt.step()
        tr_sum += loss.item() * xb.shape[0]
        tr_n += xb.shape[0]

    hyper.eval()
    val_sum, val_n = 0, 0
    with torch.no_grad():
        for xb, yb in val_dl_h:
            xb = xb.to(device)
            yb = yb.to(device)
            logits, _ = hyper(xb)
            loss = sum(bce(logits[:, i], yb[:, i]) for i in range(4))
            val_sum += loss.item() * xb.shape[0]
            val_n += xb.shape[0]

    print({'epoch': epoch, 'train_loss': tr_sum / tr_n, 'val_loss': val_sum / val_n})

{'epoch': 0, 'train_loss': 1.5702309160232544, 'val_loss': 1.1107722491663894}
{'epoch': 1, 'train_loss': 0.9733332430521647, 'val_loss': 0.9669476207972302}
{'epoch': 2, 'train_loss': 0.8650137734413147, 'val_loss': 0.876959917742178}
{'epoch': 3, 'train_loss': 0.7545253596305848, 'val_loss': 0.7901923321073573}
{'epoch': 4, 'train_loss': 0.7517554936408997, 'val_loss': 0.8061016397009567}
{'epoch': 5, 'train_loss': 0.717071817557017, 'val_loss': 0.9937234759695304}
{'epoch': 6, 'train_loss': 0.714088129679362, 'val_loss': 0.7729791096591074}
{'epoch': 7, 'train_loss': 0.6836002349853516, 'val_loss': 0.7607873014718385}
{'epoch': 8, 'train_loss': 0.6417098573048909, 'val_loss': 0.7372476916065275}
{'epoch': 9, 'train_loss': 0.6601831353505453, 'val_loss': 0.730190442607308}
{'epoch': 10, 'train_loss': 0.6259102507432301, 'val_loss': 0.7291919058616009}
{'epoch': 11, 'train_loss': 0.6407708304723104, 'val_loss': 0.948774862726894}
{'epoch': 12, 'train_loss': 0.6480291004180908, 'val_lo

In [36]:
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

@torch.no_grad()
def calc_metrics(model, X, Y, name):
    logits = model(torch.from_numpy(X).to(device))
    logits = logits[0] if isinstance(logits, tuple) else logits
    probs = torch.sigmoid(logits).cpu().numpy()
    pred = (probs >= 0.5).astype(int)

    rows = []
    for j, q in enumerate(["Q1", "Q2", "Q3", "Q4"]):
        y = Y[:, j]
        rows.append({
            "model": name,
            "question": q,
            "auroc": roc_auc_score(y, probs[:, j]),
            "balanced_accuracy": balanced_accuracy_score(y, pred[:, j]),
        })
    return pd.DataFrame(rows)

In [37]:
metrics_hyper = calc_metrics(hyper, X_val_h, Y_val_h, "Hyper-PCD")
display(metrics_hyper)


/home/seankopylov/.venv/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/seankopylov/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


,model,question,auroc,balanced_accuracy
0,Hyper-PCD,Q1,1.000000,1.000000
1,Hyper-PCD,Q2,0.937606,0.844897
2,Hyper-PCD,Q3,0.879262,0.794450
3,Hyper-PCD,Q4,NaN,1.000000


## Part 5

In [ ]:
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

class StaticMLP(nn.Module):
    def __init__(self, h_dim=768, q_dim=16):
        super().__init__()
        self.q = nn.Embedding(4, q_dim)
        self.net = nn.Sequential(nn.Linear(h_dim + q_dim, 64), nn.GELU(), nn.Linear(64, 1))

    def forward(self, h):
        b = h.shape[0]
        q = self.q.weight.unsqueeze(0).expand(b, -1, -1)
        hh = h.unsqueeze(1).expand(-1, 4, -1)
        return self.net(torch.cat([hh, q], dim=2)).squeeze(-1)

def train(model, train_dl, val_dl, epochs=15, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    bce = nn.BCEWithLogitsLoss()
    for epoch in range(epochs):
        model.train()
        tr_sum, tr_n = 0, 0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            out = model(xb)
            loss = sum(bce(out[:, i], yb[:, i]) for i in range(4))
            loss.backward()
            opt.step()
            tr_sum += loss.item() * xb.shape[0]
            tr_n += xb.shape[0]

        model.eval()
        val_sum, val_n = 0.0, 0
        with torch.no_grad():
            for xb, yb in val_dl:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                loss = sum(bce(out[:, i], yb[:, i]) for i in range(4))
                val_sum += loss.item() * xb.shape[0]
                val_n += xb.shape[0]
        print({'epoch': epoch, 'train_loss': tr_sum / tr_n, 'val_loss': val_sum / val_n})
    return model

In [25]:
static = train(StaticMLP(), train_dl_h, val_dl_h, epochs=15, lr=1e-3)

{'epoch': 0, 'train_loss': 1.9221780344645183, 'val_loss': 1.3543953815366878}
{'epoch': 1, 'train_loss': 1.1401211824417115, 'val_loss': 1.0842041626618177}
{'epoch': 2, 'train_loss': 0.9552940468788147, 'val_loss': 0.9969052812739614}
{'epoch': 3, 'train_loss': 0.8643651138941447, 'val_loss': 0.9174761804965658}
{'epoch': 4, 'train_loss': 0.8093379085858663, 'val_loss': 0.8488297695778196}
{'epoch': 5, 'train_loss': 0.7590676124890645, 'val_loss': 0.8216234620558013}
{'epoch': 6, 'train_loss': 0.7320497713088989, 'val_loss': 0.8060818976218548}
{'epoch': 7, 'train_loss': 0.6906308666865031, 'val_loss': 0.793562281022378}
{'epoch': 8, 'train_loss': 0.705646198908488, 'val_loss': 0.8255913137295924}
{'epoch': 9, 'train_loss': 0.6764828958511353, 'val_loss': 0.7768001013210425}
{'epoch': 10, 'train_loss': 0.6595428760846456, 'val_loss': 0.7637076301312228}
{'epoch': 11, 'train_loss': 0.6732652727762858, 'val_loss': 0.8500978013426521}
{'epoch': 12, 'train_loss': 0.6880777921676636, 'val

In [38]:
metrics_static = calc_metrics(static, X_val_h, Y_val_h, "static-mlp")
display(metrics_static)

/home/seankopylov/.venv/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/seankopylov/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


,model,question,auroc,balanced_accuracy
0,static-mlp,Q1,1.000000,1.000000
1,static-mlp,Q2,0.933714,0.828842
2,static-mlp,Q3,0.877354,0.770022
3,static-mlp,Q4,NaN,1.000000


In [27]:
class DenseHyper(nn.Module):
    def __init__(self, h_dim=768, q_dim=16):
        super().__init__()
        self.q = nn.Embedding(4, q_dim)
        self.hyper = nn.Sequential(nn.Linear(h_dim, 64), nn.GELU(), nn.Linear(64, 145))

    def forward(self, h):
        p = self.hyper(h)
        w1 = p[:, :128].view(-1, 8, 16)
        b1 = p[:, 128:136].view(-1, 1, 8)
        w2 = p[:, 136:144].view(-1, 1, 8)
        b2 = p[:, 144:].view(-1, 1)
        q = self.q.weight.unsqueeze(0)
        h1 = torch.matmul(q, w1.transpose(1, 2)) + b1
        h1 = F.gelu(h1)
        return (h1 * w2).sum(dim=2) + b2

dense = train(DenseHyper(), train_dl_h, val_dl_h, epochs=15, lr=1e-3)

{'epoch': 0, 'train_loss': 1.2559402481714885, 'val_loss': 1.0282786585139936}
{'epoch': 1, 'train_loss': 0.8754162613550822, 'val_loss': 0.8878000832478935}
{'epoch': 2, 'train_loss': 0.8390110143025716, 'val_loss': 0.895895994766772}
{'epoch': 3, 'train_loss': 0.7770758956273397, 'val_loss': 1.027980115435539}
{'epoch': 4, 'train_loss': 0.7662751119931539, 'val_loss': 0.8141151535401651}
{'epoch': 5, 'train_loss': 0.6751968421936035, 'val_loss': 0.8607175711098067}
{'epoch': 6, 'train_loss': 0.6751057175000509, 'val_loss': 0.8280768255939542}
{'epoch': 7, 'train_loss': 0.6921456460952758, 'val_loss': 0.7845865794278066}
{'epoch': 8, 'train_loss': 0.6236196306546529, 'val_loss': 0.7687068094900988}
{'epoch': 9, 'train_loss': 0.6511402103106181, 'val_loss': 0.8546289376891717}
{'epoch': 10, 'train_loss': 0.6612867021560669, 'val_loss': 0.7681505184290242}
{'epoch': 11, 'train_loss': 0.6146129369735718, 'val_loss': 0.7170817301543116}
{'epoch': 12, 'train_loss': 0.5948741629918416, 'val

In [39]:
metrics_dense = calc_metrics(dense, X_val_h, Y_val_h, "Dense-Hyper")
display(metrics_dense)

/home/seankopylov/.venv/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/seankopylov/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


,model,question,auroc,balanced_accuracy
0,Dense-Hyper,Q1,1.000000,1.000000
1,Dense-Hyper,Q2,0.939136,0.861239
2,Dense-Hyper,Q3,0.880766,0.797237
3,Dense-Hyper,Q4,NaN,1.000000


## Final part

In [ ]:
def hyper_decode(model, z):
    p = model.hyper(z)
    w1 = p[:, :128].view(-1, 8, 16)
    b1 = p[:, 128:136].view(-1, 1, 8)
    w2 = p[:, 136:144].view(-1, 1, 8)
    b2 = p[:, 144:].view(-1, 1)
    q = model.q.weight.unsqueeze(0)
    h1 = torch.matmul(q, w1.transpose(1, 2)) + b1
    h1 = F.gelu(h1)
    return (h1 * w2).sum(dim=2) + b2

@torch.no_grad()
def get_logits(model, X, ablate_idx=None, return_z=False, bs=256):
    dl = DataLoader(TensorDataset(torch.from_numpy(X)), batch_size=bs, shuffle=False, num_workers=0)
    model.eval()
    L, Z = [], []
    for (xb,) in dl:
        xb = xb.to(device)
        logits, z = model(xb)
        if ablate_idx is not None:
            z = z.clone()
            z[:, ablate_idx] = 0
            logits = hyper_decode(model, z)
        if return_z:
            Z.append(z.cpu().numpy())
        L.append(logits.cpu().numpy())
    if return_z:
        return np.concatenate(L), np.concatenate(Z)
    return np.concatenate(L)

logits_intro, z_intro = get_logits(hyper, X_intro, return_z=True)
dead_concepts = int((z_intro.max(axis=0) <= 0).sum())

In [45]:
def calc_metrics_from_logits(logits, Y, name):
    probs = 1 / (1 + np.exp(-logits))
    pred = (probs >= 0.5).astype(int)
    rows = []
    for j, q in enumerate(["Q1", "Q2", "Q3", "Q4"]):
        y = Y[:, j].astype(int)
        auroc = roc_auc_score(y, probs[:, j])
        rows.append({
            "model": name,
            "question": q,
            "auroc": auroc,
            "balanced_accuracy": balanced_accuracy_score(y, pred[:, j]),
        })
    return pd.DataFrame(rows)

def corr_abs(a, b):
    if a.std() == 0 or b.std() == 0:  # it's important! 
        return 0.0
    c = np.corrcoef(a, b)[0, 1]
    return 0.0 if np.isnan(c) else float(abs(c))

corr_q1 = np.array([corr_abs(z_intro[:, j], Y_intro[:, 0]) for j in range(z_intro.shape[1])])
corr_q4 = np.array([corr_abs(z_intro[:, j], Y_intro[:, 3]) for j in range(z_intro.shape[1])])
top3_q1 = np.argsort(corr_q1)[-3:][::-1]
top3_q4 = np.argsort(corr_q4)[-3:][::-1]
ablate_idx = int(top3_q1[0])

logits_h_abl = get_logits(hyper, X_val_h, ablate_idx=ablate_idx)

tbl_abl = calc_metrics_from_logits(logits_h_abl, Y_val_h, "Hyper-PCD-ablation")

display(tbl_abl)


/home/seankopylov/.venv/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/seankopylov/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


,model,question,auroc,balanced_accuracy
0,Hyper-PCD-ablation,Q1,0.999900,0.974771
1,Hyper-PCD-ablation,Q2,0.937382,0.844897
2,Hyper-PCD-ablation,Q3,0.877706,0.791838
3,Hyper-PCD-ablation,Q4,NaN,1.000000


In [47]:
diag_tbl = pd.DataFrame({
    'metric': ['dead_concepts', 'top3_q1', 'top3_q4', 'ablate_idx_q1'],
    'value': [dead_concepts, top3_q1.tolist(), top3_q4.tolist(), ablate_idx]
})

print('Hyper-PCD diagnostics + ablation:')
display(diag_tbl)

Hyper-PCD diagnostics + ablation:


,metric,value
0,dead_concepts,104
1,top3_q1,"[66, 41, 45]"
2,top3_q4,"[13, 85, 100]"
3,ablate_idx_q1,66
